# 03 — Model Comparison

Compare Kronos (pre-trained LLM) vs LightGBM (traditional ML) signals:
- Load or generate predictions from both models
- IC and Rank IC comparison
- NAV overlay (backtest both signals)
- Evaluation metrics table

In [ ]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["figure.dpi"] = 120

print(f"Project root: {PROJECT_ROOT}")

## 1. Load Predictions

Load pre-computed predictions saved by `scripts/predict_lightgbm.py` and `scripts/predict_kronos.py`. Adjust paths if needed.

In [ ]:
from big_a.qlib_config import init_qlib

init_qlib()

# Define prediction file paths (relative to project root)
lgbm_pred_path = PROJECT_ROOT / "output" / "lightgbm_predictions.parquet"
kronos_pred_path = PROJECT_ROOT / "output" / "kronos_predictions.csv"

predictions = {}

# Load LightGBM predictions
if lgbm_pred_path.exists():
    lgbm_df = pd.read_parquet(lgbm_pred_path)
    predictions["LightGBM"] = lgbm_df
    print(f"LightGBM predictions: {lgbm_df.shape}")
else:
    print(f"LightGBM predictions not found at {lgbm_pred_path}")
    print("  Run: uv run python scripts/predict_lightgbm.py")

# Load Kronos predictions
if kronos_pred_path.exists():
    kronos_df = pd.read_csv(kronos_pred_path, index_col=[0, 1])
    predictions["Kronos"] = kronos_df
    print(f"Kronos predictions: {kronos_df.shape}")
else:
    print(f"Kronos predictions not found at {kronos_pred_path}")
    print("  Run: uv run python scripts/predict_kronos.py")

if not predictions:
    print("\n⚠ No prediction files found. Generate predictions first using the CLI scripts.")

## 2. IC / Rank IC Comparison

In [ ]:
from qlib.data import D
from big_a.backtest.evaluation import calc_ic, calc_rank_ic, calc_icir

# Compute actual forward returns for IC calculation
# Use the intersection of instruments across all prediction sets
all_instruments = set()
for pred_df in predictions.values():
    if hasattr(pred_df.index, 'get_level_values'):
        all_instruments.update(pred_df.index.get_level_values(1).unique())

if all_instruments and predictions:
    instruments_list = list(all_instruments)[:100]
    # Get the date range from predictions
    sample_pred = next(iter(predictions.values()))
    dates = sample_pred.index.get_level_values(0)
    start = str(dates.min().date()) if hasattr(dates.min(), 'date') else str(dates.min())
    end = str(dates.max().date()) if hasattr(dates.max(), 'date') else str(dates.max())

    label_expr = ["Ref($close, -2) / Ref($close, -1) - 1"]
    actual = D.features(
        instruments_list,
        fields=label_expr,
        start_time=start,
        end_time=end,
    )
    actual.columns = ["score"]

    ic_data = {}
    for name, pred_df in predictions.items():
        ic = calc_ic(pred_df, actual)
        rank_ic = calc_rank_ic(pred_df, actual)
        ic_data[name] = {"ic": ic, "rank_ic": rank_ic}
        print(f"{name}: Mean IC={ic.mean():.4f}, Mean Rank IC={rank_ic.mean():.4f}, ICIR={calc_icir(ic):.4f}")
else:
    print("No predictions loaded — skipping IC analysis.")

In [ ]:
# Plot IC time series for each model
if 'ic_data' in dir() and ic_data:
    fig, axes = plt.subplots(len(ic_data), 1, figsize=(14, 4 * len(ic_data)), squeeze=False)
    for i, (name, data) in enumerate(ic_data.items()):
        ax = axes[i, 0]
        ic = data["ic"]
        ax.bar(ic.index, ic.values, width=0.8, alpha=0.6, label="IC")
        ax.axhline(ic.mean(), color="red", linestyle="--", label=f"Mean={ic.mean():.4f}")
        ax.axhline(0, color="black", linewidth=0.5)
        ax.set_title(f"{name} — IC Time Series")
        ax.set_ylabel("IC")
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 3. NAV Overlay (Backtest Both Signals)

In [ ]:
from big_a.backtest.engine import run_backtest

nav_curves = {}

for name, pred_df in predictions.items():
    try:
        report, positions = run_backtest(pred_df)
        cum_nav = (1 + report["return"]).cumprod()
        cum_bench = (1 + report["bench"]).cumprod()
        nav_curves[name] = {"strategy": cum_nav, "benchmark": cum_bench}
        print(f"{name}: {len(report)} trading days, final NAV={cum_nav.iloc[-1]:.4f}")
    except Exception as e:
        print(f"{name} backtest failed: {e}")

if nav_curves:
    fig, ax = plt.subplots(figsize=(14, 6))
    for name, curves in nav_curves.items():
        ax.plot(curves["strategy"].index, curves["strategy"].values, label=f"{name} Strategy", linewidth=1.2)

    # Plot benchmark once
    bench = next(iter(nav_curves.values()))["benchmark"]
    ax.plot(bench.index, bench.values, label="Benchmark", linewidth=1, alpha=0.7, color="gray")

    ax.set_title("NAV Overlay: Kronos vs LightGBM")
    ax.set_ylabel("NAV")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 4. Evaluation Metrics Table

In [ ]:
from big_a.backtest.evaluation import compare_models

if predictions and 'actual' in dir():
    comparison = compare_models(predictions, actual)
    print("\n=== Model Comparison Table ===")
    print(comparison.to_string(float_format="%.4f"))

    # Bar chart
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, metric in zip(axes, ["mean_ic", "mean_rank_ic", "icir"]):
        vals = comparison[metric]
        ax.bar(comparison.index, vals, alpha=0.7)
        ax.set_title(metric)
        ax.axhline(0, color="black", linewidth=0.5)
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("Load predictions and actual returns first to generate comparison table.")